# 09 - approach A (exploratory): tell the decoder the band and pixel size

Added 2026-09-25, **after** the one test run. In the first arm, band differences
on SI looked like resolution differences; the data cannot settle this. Does the
row-3 decoder draw better
fronts when it is told each image's radar band and ground resolution?

| variant | what it is |
|---|---|
| plain | row 3's decoder (frozen features + image skip), retrained here |
| film | the same decoder plus FiLM conditioning on band X/C/L and resolution / 20 m; identical to plain at the start of training |

Both variants of a seed start from the same decoder weights and see the same
4 000 training tiles (uniform sample) in the same order. Frozen features as in
the second arm: DINOv3 SAT layer 21, C-RADIO layer 32. Seeds 0, 1, 2.
Validation glacier **SI only**; the test glaciers are not touched here.

**Pre-registered decision rule, fixed before any number:** FiLM helps on SI if its
mean MDE over the three seeds is lower than plain's by more than the larger of
the two seed standard deviations, with no more missed fronts in total. Hurts:
the same margin the other way. Anything else: no clear effect. A later test run
of these heads would be a second look at the test glaciers and is reported as such.

**Runtime:** A100 High-RAM. Cells 1-3, then 4 (43 min satellite DINOv3, 49 min C-RADIO, recorded), then 5.
Results go to Drive as `cond_*` files; nothing from notebooks 03-08 is overwritten.

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# --- cell: secrets, GPU, data ---
import os, shutil
from pathlib import Path
import torch
from sartransfer.env import load_env, require, model_cache_on_drive
from sartransfer.data import inventory as inv

load_env()
require("HF_TOKEN", "gated model weights")
model_cache_on_drive()          # weights downloaded once, kept on Drive
assert torch.cuda.is_available(), "switch to a GPU runtime"
print("gpu:", torch.cuda.get_device_properties(0).name)
DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), Path("/content/caffe"))
DATA = inv.find_data_root(Path("/content/caffe"))
df = inv.scan(DATA, subdirs=("sar_images",))
print(len(df), "SAR images | free disk:", f"{shutil.disk_usage('/content').free/1e9:.0f} GB")

In [4]:
# --- cell: APPROACH A on SI -- plain vs FiLM decoder, 3 seeds, both encoders (43-49 min per encoder, recorded) ---
import time
from sartransfer.conditioning import run_conditioning

LAYER = {"dinov3-l-sat": 21, "cradio-v4-h": 32}      # same layers as the second arm
for encoder, layer in LAYER.items():
    t0 = time.perf_counter()
    try:
        run_conditioning(encoder, "caffe", layer, df, DATA, RES, token=os.environ["HF_TOKEN"])
    except Exception as e:
        print(f"FAILED {encoder}: {type(e).__name__}: {str(e)[:300]}")
    print(f"{encoder}: {(time.perf_counter() - t0) / 60:.0f} min\n")


===== conditioning dinov3-l-sat__caffe__tile512__L21 on val: [('plain', 0), ('plain', 1), ('plain', 2), ('film', 0), ('film', 1), ('film', 2)] =====


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('train', 'val'): 559 of 681 images
  100/559  0.7 min
  200/559  1.3 min
  300/559  2.4 min
  400/559  3.3 min
  500/559  3.6 min
features ready in 4.0 min | 4000 of 12595 training tiles kept | 121 val images
    step  1000  loss 0.3116  1.2 min
    step  2000  loss 0.1554  2.5 min
    step  3000  loss 0.1122  3.7 min
    step  4000  loss 0.1025  4.9 min
  plain s0   544.4 k params | MDE   1453 m (0 no front) | 4-class pixel mIoU 0.748 | 6.3 min
    step  1000  loss 0.3145  1.2 min
    step  2000  loss 0.1302  2.4 min
    step  3000  loss 0.1097  3.6 min
    step  4000  loss 0.0935  4.9 min
  plain s1   544.4 k params | MDE   1390 m (0 no front) | 4-class pixel mIoU 0.755 | 6.2 min
    step  1000  loss 0.3170  1.2 min
    step  2000  loss 0.1416  2.4 min
    step  3000  loss 0.1126  3.6 min
    step  4000  loss 0.0962  4.9 min
  plain s2   544.4 k params | MDE   1458 m (0 no front) | 4-class pixel mIoU 0.751 | 6.2 min
    st

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

cradio-v4-h: 651.6 M parameters loaded
restricted to splits ('train', 'val'): 559 of 681 images
  100/559  0.7 min
  200/559  1.4 min
  300/559  2.6 min
  400/559  3.6 min
  500/559  3.9 min
features ready in 4.4 min | 4000 of 12595 training tiles kept | 121 val images
    step  1000  loss 0.3225  1.4 min
    step  2000  loss 0.1588  2.7 min
    step  3000  loss 0.1191  4.1 min
    step  4000  loss 0.1054  5.4 min
  plain s0   569.0 k params | MDE   1627 m (0 no front) | 4-class pixel mIoU 0.697 | 6.8 min
    step  1000  loss 0.3146  1.4 min
    step  2000  loss 0.1532  2.7 min
    step  3000  loss 0.1212  4.1 min
    step  4000  loss 0.1057  5.4 min
  plain s1   569.0 k params | MDE   1571 m (0 no front) | 4-class pixel mIoU 0.715 | 6.8 min
    step  1000  loss 0.3287  1.4 min
    step  2000  loss 0.1472  2.7 min
    step  3000  loss 0.1194  4.1 min
    step  4000  loss 0.1004  5.5 min
  plain s2   569.0 k params | MDE   1545 m (1 no front) | 4-class pixel mIoU 0.705 | 6.9 min
    ste

In [5]:
# --- cell: RESULTS -- plain vs FiLM on SI over three seeds, with the pre-registered verdict ---
import pandas as pd
from sartransfer.conditioning import conditioning_tables

agg, per_sensor = conditioning_tables(RES, "val")
pd.set_option("display.width", 220, "display.precision", 3)
print("=== SI: MDE (m) over 3 seeds, plain vs FiLM ===")
print(agg.drop(columns="verdict").round(3).to_string(index=False))
print()
for run, verdict in agg.groupby("run").verdict.first().items():
    print(f"{run}: {verdict}")
print("\n=== SI: MDE per sensor @ resolution (m), mean over seeds ===")
print(per_sensor.to_string())
print("\nreference on SI (second arm, row 3 as reported): DINOv3 SAT 1,329 +- 25 m; "
      "C-RADIO 1,244 +- 56 m; U-Net from scratch 1,732 m")

=== SI: MDE (m) over 3 seeds, plain vs FiLM ===
                              run variant  seeds  MDE_mean  MDE_sd  no_front_total  mIoU4_mean  params
 cradio-v4-h__caffe__tile512__L32    film      3  1507.371  82.528               0       0.705  591300
 cradio-v4-h__caffe__tile512__L32   plain      3  1580.963  42.189               1       0.706  568964
dinov3-l-sat__caffe__tile512__L21    film      3  1374.415 105.680               0       0.756  566724
dinov3-l-sat__caffe__tile512__L21   plain      3  1433.856  37.771               0       0.751  544388

cradio-v4-h__caffe__tile512__L32: no clear effect: difference +74 m is within the seed spread (83 m)
dinov3-l-sat__caffe__tile512__L21: no clear effect: difference +59 m is within the seed spread (106 m)

=== SI: MDE per sensor @ resolution (m), mean over seeds ===
group                                      ENVISAT@20m  ERS@20m  PALSAR@17m  RSAT@12m  RSAT@20m  TSX@7m
run                               variant                         